In [1]:
# import os
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--conf "spark.driver.extraJavaOptions=-Djdk.security.allowTemporaryOverride=true" pyspark-shell'

In [5]:
import pyspark

In [6]:
pyspark.__file__

'/usr/local/python/3.12.1/lib/python3.12/site-packages/pyspark/__init__.py'

In [16]:
spark.version

'4.1.1'

In [8]:
from pyspark.sql import SparkSession

In [9]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/15 21:18:55 WARN Utils: Your hostname, codespaces-09f4d1, resolves to a loopback address: 127.0.0.1; using 10.0.1.81 instead (on interface eth0)
26/03/15 21:18:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/15 21:18:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [10]:
#!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

In [11]:
#!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

In [12]:
#!head data/taxi_zone_lookup.csv

In [13]:
from pyspark.sql import types

yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.LongType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.LongType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.LongType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [14]:
# df = spark.read \
#     # .option("header", "true") \
#     .parquet('data/yellow_tripdata_2025-11.parquet') \
#     .schema(yellow_schema) \
df = spark.read \
        .schema(yellow_schema) \
        .parquet('data/yellow_tripdata_2025-11.parquet') \
        .repartition(4)
    # .schema(yellow_schema)
    # .parquet('data/yellow_tripdata_2025-11.parquet')
    # .repartition(4)


In [15]:
df.show(10)

[Stage 0:=============================>                             (1 + 1) / 2]

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       2| 2025-11-18 17:49:34|  2025-11-18 18:15:35|              1|         1.92|         1|                 N|         163|         107|           1|       21.9|  2.5|    0.5|      5.83|         0.0|                  1.0

In [22]:
df.write.mode("overwrite").parquet('data/output_data/')

In [30]:
!ls -lh data/output_data/

total 108M
-rw-r--r-- 1 codespace codespace   0 Mar 15 21:38 _SUCCESS
-rw-r--r-- 1 codespace codespace 27M Mar 15 21:38 part-00000-fc2febe8-fbe1-48b9-a158-181c64ed8c86-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 27M Mar 15 21:38 part-00001-fc2febe8-fbe1-48b9-a158-181c64ed8c86-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 27M Mar 15 21:38 part-00002-fc2febe8-fbe1-48b9-a158-181c64ed8c86-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 27M Mar 15 21:38 part-00003-fc2febe8-fbe1-48b9-a158-181c64ed8c86-c000.snappy.parquet


In [34]:
df.registerTempTable('yellow')

/usr/local/python/3.12.1/lib/python3.12/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [35]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [55]:
spark.sql("""
SELECT
    
    count(1)
FROM
    yellow
Where 
    CAST(tpep_pickup_datetime AS DATE) = '2025-11-15' 
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [39]:

spark.sql("""
SELECT 
        VendorID,
        tpep_pickup_datetime,
        tpep_dropoff_datetime,
        (timestampdiff(SECOND, tpep_pickup_datetime, tpep_dropoff_datetime) / 3600) AS duration_hours
FROM 
        yellow

ORDER BY 
        duration_hours DESC 
""").show()


[Stage 25:=============================>                            (2 + 2) / 4]

+--------+--------------------+---------------------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|    duration_hours|
+--------+--------------------+---------------------+------------------+
|       2| 2025-11-26 20:22:12|  2025-11-30 15:01:00| 90.64666666666666|
|       2| 2025-11-27 04:22:41|  2025-11-30 09:19:35| 76.94833333333334|
|       2| 2025-11-03 10:42:55|  2025-11-06 14:55:45| 76.21388888888889|
|       2| 2025-11-07 11:23:22|  2025-11-10 08:40:41| 69.28861111111111|
|       2| 2025-11-18 17:12:47|  2025-11-21 12:17:37| 67.08055555555555|
|       2| 2025-11-22 17:45:30|  2025-11-25 09:07:36| 63.36833333333333|
|       1| 2025-11-01 07:44:57|  2025-11-03 16:07:53|56.382222222222225|
|       2| 2025-11-27 14:34:56|  2025-11-29 14:43:48|48.147777777777776|
|       1| 2025-11-01 14:55:34|  2025-11-03 14:24:16| 47.47833333333333|
|       1| 2025-11-22 16:05:20|  2025-11-24 13:31:59| 45.44416666666667|
|       2| 2025-11-26 13:20:12|  2025-11-28 09:22:4

In [41]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('data/taxi_zone_lookup.csv')

In [42]:
df_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [43]:
df_zones.printSchema()

root
 |-- LocationID: string (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [51]:
zones_schema = types.StructType([
    types.StructField('LocationID', types.IntegerType(), True),
    types.StructField('Borough', types.StringType(), True),
    types.StructField('Zone', types.StringType(), True),
    types.StructField('service_zone', types.StringType(), True),
])

In [52]:
df_zones = spark.read \
    .option("header", "true") \
    .schema(zones_schema) \
    .csv('data/taxi_zone_lookup.csv')

In [53]:
df_zones.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [54]:
df_zones.createOrReplaceTempView('zones')

In [60]:
spark.sql("""
SELECT 
        
      
        Zone,
        count(1) as trips_numbers
FROM 
        yellow
left join 
        zones
on yellow.PULocationID = zones.LocationID

group by Zone

ORDER BY 
        trips_numbers asc 
""").show()

[Stage 47:=============================>                            (2 + 2) / 4]

+--------------------+-------------+
|                Zone|trips_numbers|
+--------------------+-------------+
|Governor's Island...|            1|
|Eltingville/Annad...|            1|
|       Arden Heights|            1|
|       Port Richmond|            3|
|   Rossville/Woodrow|            4|
|         Great Kills|            4|
| Green-Wood Cemetery|            4|
|       Rikers Island|            4|
|         Jamaica Bay|            5|
|         Westerleigh|           12|
|New Dorp/Midland ...|           14|
|       West Brighton|           14|
|             Oakwood|           14|
|        Crotona Park|           14|
|       Willets Point|           15|
|Breezy Point/Fort...|           16|
|Saint George/New ...|           17|
|       Broad Channel|           18|
|     Mariners Harbor|           21|
|Heartland Village...|           22|
+--------------------+-------------+
only showing top 20 rows
